In [1]:
import os
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from PIL import Image
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

In [2]:
 
train_csv = os.path.expanduser("~/Desktop/kilns/train/data.csv")
train_img_dir = os.path.expanduser("~/Desktop/kilns/train")
test_img_dir  = os.path.expanduser("~/Desktop/kilns/test")

In [3]:
#data augmentation part , edit later for improvement

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


In [4]:
class KilnTrainDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform

        # Figure out which column has the image id
        if "filename" in self.df.columns:
            self.name_col = "filename"
        elif "index" in self.df.columns:
            self.name_col = "index"
        else:
            raise ValueError("CSV must contain either 'filename' or 'index' column.")

        if "label" not in self.df.columns:
            raise ValueError("CSV must contain a 'label' column (0/1).")

        print("Train CSV columns:", self.df.columns.tolist())
        print("Using image id column:", self.name_col)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = str(row[self.name_col])
        label = int(row["label"])

        if not fname.endswith(".png"):
            fname = fname + ".png"

        img_path = os.path.join(self.img_dir, fname)
        if not os.path.exists(img_path):
            raise FileNotFoundError(img_path)

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        return img, label


class KilnTestDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.files = sorted(
            f for f in os.listdir(self.img_dir)
            if f.endswith(".png") and " " not in f
        )
        print("Number of test images:", len(self.files))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img_path = os.path.join(self.img_dir, fname)
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

       
        img_id = os.path.splitext(fname)[0]
        return img, img_id


In [5]:
train_ds = KilnTrainDataset(train_csv, train_img_dir, transform=preprocess)
test_ds  = KilnTestDataset(test_img_dir, transform=preprocess)

len(train_ds), len(test_ds)


Train CSV columns: ['index', 'label']
Using image id column: index
Number of test images: 724


(1617, 724)

In [6]:
import torch
torch.multiprocessing.set_start_method("fork", force=True)


In [7]:
num_workers=0
persistent_workers=False


In [8]:
batch_size = 32

train_loader = DataLoader(train_ds, batch_size=batch_size,
                          shuffle=True, num_workers=0, persistent_workers=False)
test_loader = DataLoader(test_ds, batch_size=batch_size,
                         shuffle=False, num_workers=0, persistent_workers=False)


In [11]:
weights = EfficientNet_B0_Weights.IMAGENET1K_V1
base_model = efficientnet_b0(weights=weights)

# Freeze backbone
for param in base_model.features.parameters():
    param.requires_grad = False

# Replace classifier head
in_features = base_model.classifier[1].in_features

base_model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(in_features, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 2)   # 2 classes: 0 / 1
)

model = base_model.to(device)


In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                             lr=1e-4)

epochs = 8

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.4f}")


KeyboardInterrupt: 

In [19]:
model.eval()
all_preds = []
all_ids = []

with torch.no_grad():
    for imgs, ids in test_loader:
        imgs = imgs.to(device)
        out = model(imgs)
        preds = out.argmax(1).cpu().tolist()

        all_preds.extend(preds)
        all_ids.extend(ids)


In [20]:
import pandas as pd

df_sub = pd.DataFrame({
    "index": all_ids,
    "score": all_preds
})

df_sub


,index,score
0,K1617,1
1,K1618,1
2,K1619,1
3,K1620,1
4,K1621,1
...,...,...
719,K2336,0
720,K2337,0
721,K2338,1
722,K2339,1


In [21]:
df_sub.to_csv("efficientNet4.csv", index=False)
 

trying the raw pretrained model without finetuning


In [13]:
import torch
import torch.nn as nn
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"

model = models.efficientnet_b0(pretrained=True)

# Replace classifier head
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 2)  # 2 classes: 0 and 1

model = model.to(device)
 

/Users/hazelzhao/miniforge3/envs/kiln/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/hazelzhao/miniforge3/envs/kiln/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [14]:
preprocess = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [16]:
import torch
print(torch.backends.mps.is_available())
print(torch.backends.mps.is_built())


True
True


In [17]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)


Using device: mps


In [18]:
model = model.to(device)


In [20]:
epochs = 8

for epoch in range(epochs):
    model.train()
    total = 0
    correct = 0
    running_loss = 0.0

    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/total:.4f} - Acc: {correct/total:.4f}")


Epoch 1/8 - Loss: 0.3707 - Acc: 0.8578
Epoch 2/8 - Loss: 0.1829 - Acc: 0.9419
Epoch 3/8 - Loss: 0.0864 - Acc: 0.9814
Epoch 4/8 - Loss: 0.0617 - Acc: 0.9827
Epoch 5/8 - Loss: 0.0423 - Acc: 0.9889
Epoch 6/8 - Loss: 0.0281 - Acc: 0.9938
Epoch 7/8 - Loss: 0.0192 - Acc: 0.9951
Epoch 8/8 - Loss: 0.0182 - Acc: 0.9938


In [21]:
model.eval()
all_preds = []
all_ids = []

with torch.no_grad():
    for imgs, ids in test_loader:
        imgs = imgs.to(device)

        outputs = model(imgs)
        _, preds = outputs.max(1)

        all_preds.extend(preds.cpu().numpy())
        all_ids.extend(ids)

import pandas as pd
df_sub = pd.DataFrame({
    'index': all_ids,
    'score': all_preds
})

df_sub.to_csv("efficient_mps_submission.csv", index=False)
 
